# 02 — IEEE 13-node unbalanced feeder

## Objectives and engineering concept

See why single- and two-phase laterals matter, load a bundled public feeder, and compare phase-specific voltage magnitudes.

## System, assumptions, and units

The one-line is the IEEE 13-node feeder shipped with the public wheel. Voltage is compared in pu at bus 671, phase by phase; feeder impedances and operating values come from the bundled benchmark input.

## Part A — Pure OpenDSS

The feeder is redirected into OpenDSS, solved, and read back at bus 671.

## Part B — Same study with CEPT

CEPT selects the bundled IEEE13 Case and runs the unbalanced load-flow path.

## Part C — Compare and verify

The three common phase magnitudes are compared with a declared teaching tolerance. The manual feeder setup is what CEPT packages; neither side proves a user's physical network.

The IEEE 13-node feeder contains single- and two-phase laterals, so a balanced positive-sequence approximation would hide the quantity we want to teach. We load the bundled feeder directly with OpenDSS and through CEPT, then compare the three phase magnitudes at bus 671.

## Interpret, exercise, and reproduce

Interpret phase imbalance rather than averaging it away. As an exercise, inspect another feeder bus and rerun every cell from a restarted session. The bundled feeder and installed CEPT version make the input snapshot explicit.

In [ ]:
from importlib.resources import files
from pathlib import Path
import opendssdirect as dss
from cept.public import demo_case, run_study

master = Path(str(files('cept').joinpath('testsystems', 'ieee13', 'IEEE13Nodeckt.dss')))
dss.Basic.ClearAll()
dss.Basic.DataPath(str(master.parent))
dss.Text.Command(f'Redirect "{master}"')
dss.Text.Command('CalcVoltageBases')
dss.Text.Command('Solve')
assert dss.Solution.Converged()
dss.Circuit.SetActiveBus('671')
direct_values = dss.Bus.puVmagAngle()
direct_by_phase = {phase: float(direct_values[2 * (phase - 1)]) for phase in (1, 2, 3)}

run = run_study(demo_case('unbalanced-load-flow'))
assert run.verification['passed'] is True
cept_by_phase = {
    item.phase: item.v_pu
    for item in run.result.load_flow.bus_voltages
    if item.bus.lower() == '671'
}
print({'direct': direct_by_phase, 'cept': cept_by_phase})
assert set(cept_by_phase) == {1, 2, 3}
for phase in (1, 2, 3):
    assert abs(direct_by_phase[phase] - cept_by_phase[phase]) < 1e-4


Both calculations retain the feeder's phase topology. The agreement here is a public workflow regression check, not evidence that the feeder represents a user's physical project.